```yaml
title: Following NASA's Deep Space Exploration budget
description: Trace a federal account from request through outlay, then see who got the contracts.
tags: [budget, recipients, shape]
endpoints: [list_budget_accounts, get_budget_account_recipients]
```

# Following NASA's Deep Space Exploration budget

A quick tour of the budget endpoints: find a federal account, watch the dollars move through the budget lifecycle, then see which companies actually got the contracts.

## Locate the account

To explore a federal agency's budget account, you need to know the account number and fiscal year. To find the account number, you can call the `list_budget_accounts` method with a free-text `search` parameter. The `shape` parameter trims the payload to just the fields we want.

> NASA retitled this account from "Deep Space Exploration Systems" to simply "Exploration" starting in FY2024, so we search for `exploration` — a reminder that account titles drift, while the `federal_account_symbol` (here `080-0124`) is the stable key.

In [1]:
# Initialize the Tango Client
from tango import TangoClient
client = TangoClient() # Note that TANGO_API_KEY key needs to be an environment variable otherwise use TangoClient(api_key="ABC123")

In [2]:
# Search the budget endpoint
nasa = client.list_budget_accounts(
    search="exploration", fiscal_year=2024,
    shape="id,federal_account_symbol,account_title,agency_name",
).results[0]

print(nasa.federal_account_symbol, nasa.account_title, "—", nasa.agency_name)

080-0124 Exploration — National Aeronautics and Space Administration


## How the money moved

Now that we have the account number, we can look to see how much money came in and how much came out. 

One row per stage of the federal budget lifecycle — requested, enacted, apportioned, obligated, outlayed. (Note that here, the apportioned amount exceeds enacted because prior-year balances are still in play.)

In [3]:
acct = client.list_budget_accounts(
    federal_account_symbol="080-0124", fiscal_year=2024,
    shape="requested_ba,enacted_ba,apportioned,obligated_total,outlayed_total",
).results[0]

for label, key in [("Requested","requested_ba"), ("Enacted","enacted_ba"),
                   ("Apportioned","apportioned"), ("Obligated","obligated_total"),
                   ("Outlayed","outlayed_total")]:
    print(f"{label:<12} ${float(acct[key])/1e9:>6.2f}B")

Requested    $  0.00B
Enacted      $  7.65B
Apportioned  $ 15.89B
Obligated    $  7.83B
Outlayed     $  7.21B


**Nota bene**: you could have combined these two queries into a single query and just expanded the shape parameter.

## Where the money went

Finally, if you want to see which recipients received contracts under that budget account, the `get_budget_account_recipients` method joins obligations to recipients and funding offices.

In [4]:
recipients = client.get_budget_account_recipients(nasa["id"], limit=6)

for r in recipients.results:
    if not r.get("recipient"):       # skip unlinked obligations bucket
        continue
    print(r["recipient"]["legal_business_name"].title(),
          f"({r["recipient"]["uei"]})",
          "←", r["funding_office"]["office_name"].title(),
          ":", r["contract_obligated"])

The Boeing Company (SML4NN2CT556) ← Nasa Marshall Space Flight Center : 1123735127.0
Space Exploration Technologies Corp. (C6M7C2FLKER5) ← Nasa Marshall Space Flight Center : 622690296.34
Blue Origin Manufacturing, Llc (RAR2RMTVZYX3) ← Nasa Marshall Space Flight Center : 483058644.0
Lockheed Martin Corp (FYHNA5WC8XD7) ← Nasa Johnson Space Center : 402285249.46
Aerojet Rocketdyne Of De, Inc (JCZYYYQ7JGF5) ← Nasa Marshall Space Flight Center : 367286271.92


# Bonus (How much of SpaceX's money came from that account)

One thing you might want to do is see which budget accounts represented the majority of a company's contracts. 

In [5]:
from collections import defaultdict

uei = "C6M7C2FLKER5"
buckets = defaultdict(lambda: {"title": "", "agency": "", "latest_fy": 0,
                                "obligated": 0.0, "contracts": 0})

page = 1
while True:
    resp = client.get_entity_budget_flows(uei, page=page, limit=100)
    for row in resp.results:
        b = buckets[row["federal_account_symbol"]]
        if row["fiscal_year"] >= b["latest_fy"]:
            b["latest_fy"] = row["fiscal_year"]
            b["title"] = row["account"]["account_title"]
            b["agency"] = row["account"]["agency_name"]
        b["obligated"] += float(row["contract_obligated"] or 0)
        b["contracts"] += row.get("n_contracts", 0) or 0
    if not resp.next:
        break
    page += 1

total = sum(b["obligated"] for b in buckets.values())

def trunc(s, n):
    return (s[:n - 2] + "..") if len(s) > n else s

print(f"{'Account':<10} {'Agency':<30} {'Title':<32} {'Obligated':>11} {'Share':>7} {'Contracts':>10}")
print("-" * 104)
for sym, b in sorted(buckets.items(), key=lambda kv: kv[1]["obligated"], reverse=True)[:10]:
    share = b["obligated"] / total * 100 if total else 0
    print(f"{sym:<10} {trunc(b['agency'], 30):<30} {trunc(b['title'], 32):<32} "
          f"${b['obligated']/1e9:>8.2f}B {share:>6.1f}% {b['contracts']:>10}")


Account    Agency                         Title                              Obligated   Share  Contracts
--------------------------------------------------------------------------------------------------------
080-0115   National Aeronautics and Spa.. Space Operations                 $    3.78B   48.3%         40
080-0124   National Aeronautics and Spa.. Exploration                      $    3.14B   40.2%         21
057-3022   Department of Defense--Milit.. 57-3022 21/23 - Procurement, S.. $    0.36B    4.5%         19
057-3021   Department of Defense--Milit.. Space Procurement, Air Force     $    0.21B    2.7%         14
080-0120   National Aeronautics and Spa.. Science                          $    0.20B    2.6%         15
080-0131   National Aeronautics and Spa.. Space Technology                 $    0.05B    0.7%         14
097-0400   Department of Defense--Milit.. Research, Development, Test an.. $    0.04B    0.5%         11
057-3620   Department of Defense--Milit.. Research, De